In [1]:
# Import Packages
import os
import re
import sys
import time
import copy
from pathlib import Path
from IPython.display import display

import numpy as np
import pandas as pd

from itertools import combinations
from itertools import product

from sklearn.utils.parallel import Parallel, delayed

%run Model_functions.ipynb

In [2]:
#BASE_DIR = "../../DATA/AGB_DATA/Merged_Data/"
BASE_DIR = "../../Data/"

In [3]:
def create_interact_terms(X, interact_col_names):
    for col in interact_col_names:
        if col in X.columns:
            X[f'height_x_{col}'] = X['height'] * X[col]

    return X

In [4]:
def generate_feature_combinations(feature_groups, categ_cols, required_groups=None):
    if required_groups is None:
        required_groups = []

    combos = []
    group_names = list(feature_groups.keys())

    for r in range(1, len(group_names) + 1):
        for selected_groups in combinations(group_names, r):

            # Skip useless combos
            if all(g in categ_cols for g in selected_groups):
                continue
            
            features = []
            for group in selected_groups:
                features.extend(feature_groups[group])

            features = list(dict.fromkeys(features))

            combos.append({
                "group_combo": selected_groups,
                "features": features,
                "num_features": len(features)
            })

    return combos

# EMIT + CANOPY DATA

In [5]:
EMIT_DATA_CSV = BASE_DIR + "/EMIT_AGB/AGB_EMIT_EO_CANOPY.csv"

emit_raw_df = pd.read_csv(EMIT_DATA_CSV)
print(emit_raw_df.shape)

assert len(emit_raw_df["simard_height_m"].head())
assert len(emit_raw_df["tandemx_height_m"].head())

(3880, 316)


## Handle EMIT columns with NULL data

In [6]:
emit_raw_df = handle_null_columns(emit_raw_df)

Total NULL count           : 17604
Rows with at least one NULL: 1467
Total rows                 : 3880
Percentage                 : 37.8%

NULL count per column in affected rows:
EMIT_R1357    1467
EMIT_R1796    1467
EMIT_R1789    1467
EMIT_R1781    1467
EMIT_R1774    1467
EMIT_R1432    1467
EMIT_R1424    1467
EMIT_R1417    1467
EMIT_R1350    1467
EMIT_R1342    1467
EMIT_R1335    1467
EMIT_R1327    1467
dtype: int64
Dropping 12 columns:
['EMIT_R1327', 'EMIT_R1335', 'EMIT_R1342', 'EMIT_R1350', 'EMIT_R1357', 'EMIT_R1417', 'EMIT_R1424', 'EMIT_R1432', 'EMIT_R1774', 'EMIT_R1781', 'EMIT_R1789', 'EMIT_R1796']

NULL count after dropping: 0


## Feature selection

In [7]:
emit_non_feature_cols = [
    'dataset',             # metadata
    'EMIT_selected_date',  # metadata
    'EMIT_granule',        # metadata
    'start_date',          # metadata
    'end_date',            # metadata
    'capture_start',       # metadata
    'capture_end',         # metadata
    'latitude',            # coordinate
    'longitude',           # coordinate
    'plant_AGB_kg',        # Target variable
    'diameter',            # Allometric
    'height'               # Allometric
]
emit_feature_cols = [c for c in emit_raw_df.columns if c not in emit_non_feature_cols]

target = 'plant_AGB_kg'
y_emit = emit_raw_df[target]

In [8]:
EMIT_interact_cols = ['EVI', 'MSI', 'NBR', 'NDRE1', 'NDRE2', 'NDRE3', 'CIrededge']
emit_raw_df = create_interact_terms(emit_raw_df, EMIT_interact_cols)

### Feature selection with Simard canopy height

In [9]:
X_emit_t = emit_raw_df[emit_feature_cols].copy()

emit_categ_cols = get_categorical_cols(X_emit_t)

# Select TANDEMX
X_emit_t = X_emit_t.rename({'tandemx_height_m': 'height'}, axis=1)
X_emit_t = X_emit_t.drop(columns=['simard_height_m'])

### Feature selection with TandemX canopy height

In [10]:
X_emit_s = emit_raw_df[emit_feature_cols].copy()

# Select SIMARD
X_emit_s = X_emit_s.rename({'simard_height_m': 'height'}, axis=1)
X_emit_s = X_emit_s.drop(columns=['tandemx_height_m'])

## EMIT Test features

In [11]:
test_cv = 5

In [12]:
struct_features = ['height']
species         = ["species"]

emit_cols              = [c for c in X_emit_t.columns if c.startswith('EMIT')]
emit_indices           = ['EVI', 'NBR', 'MSI', 'NDVI', 'NDRE1', 'NDRE2', 'NDRE3', 'CIrededge']
emit_top_spectral      = ['EVI', 'NBR', 'MSI', 'NDRE1']
emit_interaction_terms = [c for c in X_emit_t.columns if c.startswith('height_x_')]

emit_features_list = [
    # Non-spectral baselines
    struct_features,            # height only
    species,                    # species only
    struct_features + species,  # height + species (non-spectral floor)

    # Spectral alone
    emit_cols,                  # all 158 bands
    emit_indices,               # all 8 indices
    emit_top_spectral,          # 4 curated indices

    # Spectral + height
    emit_cols    + struct_features,
    emit_indices + struct_features,
    emit_top_spectral + struct_features,

    # Spectral + height + species
    emit_cols    + struct_features + species,
    emit_indices + struct_features + species,
    emit_top_spectral + struct_features + species,

    # Spectral + height + interactions
    emit_cols + struct_features + emit_interaction_terms,
    emit_top_spectral + struct_features + emit_interaction_terms,

    # Full spectrum of features
    emit_cols + struct_features + species + emit_interaction_terms,
    emit_top_spectral + struct_features + species + emit_interaction_terms,
]

## EMIT. Create plot groups.

In [13]:
# Retain the groups/plot_id for splitting the data based on groups.
plot_groups_emit = None
if 'plot_id' in X_emit_t.columns:
    plot_groups_emit = X_emit_t['plot_id'].copy()
    X_emit_t = X_emit_t.drop(columns=['plot_id'])

if 'plot_id' in X_emit_s.columns:
    X_emit_s = X_emit_s.drop(columns=['plot_id'])

near_zero_sites_emit, high_agb_sites_emit,\
    near_zero_plots_emit, high_agb_plots_emit = \
        get_low_and_high_agb_plots(y_emit,
                                   plot_groups_emit)

grp_emit = GROUP_INFO(near_zero_sites_emit,
                      high_agb_sites_emit,
                      near_zero_plots_emit,
                      high_agb_plots_emit,
                      groups=plot_groups_emit,
                      cv=test_cv)

High-AGB threshold  : 104.49 kg
Near-zero threshold : 0.004134

Near-zero variance plots:
  Big Creek_1               : log var = 0.000036
  Big Creek_4               : log var = 0.000032
  Frenchman Caye_1          : log var = 0.000753
  Frenchman Caye_2          : log var = 0.000381
  Frenchman Caye_3          : log var = 0.000693
  Frenchman Caye_4          : log var = 0.001306
  Frenchman Caye_5          : log var = 0.001283
  Frenchman Caye_6          : log var = 0.000158
  Shipstern Lagoon_1        : log var = 0.001064
  Shipstern Lagoon_3        : log var = 0.000232
  Shipstern Lagoon_4        : log var = 0.000113
  Shipstern Lagoon_5        : log var = 0.000052
  Shipstern Lagoon_6        : log var = 0.000135

High-AGB plots:
  Channel Caye_1            : max AGB = 310.9 kg
  Channel Caye_2            : max AGB = 206.4 kg
  Channel Caye_3            : max AGB = 427.2 kg
  Channel Caye_4            : max AGB = 237.6 kg
  Channel Caye_5            : max AGB = 170.4 kg
  Channel C

In [14]:
assert plot_groups_emit is not None

# SENTINEL + CANOPY DATA

In [15]:
SENTINEL_DATA_CSV        = BASE_DIR + "/Sentinel_AGB/AGB_SENTINEL_CANOPY.csv"
sentinel_raw_df = pd.read_csv(SENTINEL_DATA_CSV)

assert len(sentinel_raw_df["simard_height_m"].head())
assert len(sentinel_raw_df["tandemx_height_m"].head())

## Sentinel feature selection

In [16]:
non_feature_cols = [
    'plant_AGB_kg',        # Target variable
    'dataset',             # metadata
    'sentinel_time',       # metadata
    'start_date',          # metadata
    'end_date',            # metadata
    'capture_start',       # metadata
    'capture_end',         # metadata
    'latitude',            # coordinate
    'longitude',           # coordinate
    'diameter',            # Allometric
    'height',               # Allometric
    'cloud_threshold_used',
    'start_date'
]

target = 'plant_AGB_kg'
y_sent = sentinel_raw_df[target]

feature_cols = [c for c in sentinel_raw_df.columns if c not in non_feature_cols]

### Feature selection with Simard canopy height

In [17]:
X_sent_t = sentinel_raw_df[feature_cols].copy()

# Select TANDEMX
X_sent_t = X_sent_t.rename({'tandemx_height_m': 'height'}, axis=1)
X_sent_t = X_sent_t.drop(columns=['simard_height_m'])

### Feature selection with Simard canopy height

In [18]:
X_sent_s = sentinel_raw_df[feature_cols].copy()

# Select SIMARD
X_sent_s = X_sent_s.rename({'simard_height_m': 'height'}, axis=1)
X_sent_s = X_sent_s.drop(columns=['tandemx_height_m'])

## Sentinel test features

In [19]:
sent_struct_features = ['height']

sent_bands   = ['Blue', 'Green', 'Red', 'RE1', 'RE2', 'RE3', 'NIR', 'SWIR1', 'SWIR2']
sent_indices = ['NDVI', 'MNDWI', 'NBR', 'EVI', 'NDRE1', 'NDRE2', 'NDRE3', 'CIrededge']

sent_top_spectral  = ['EVI', 'NIR', 'NBR', 'SWIR1']
sent_redband       = ['RE1', 'RE2', 'RE3']
sent_interaction_terms = [c for c in sentinel_raw_df.columns if c.startswith('height_x_')]

# Curated interactions matching curated spectral sets
sent_top_spectral_interactions = [f'height_x_{b}' for b in sent_top_spectral
                                  if f'height_x_{b}' in sentinel_raw_df.columns]
sent_redband_interactions      = [f'height_x_{b}' for b in sent_redband
                                  if f'height_x_{b}' in sentinel_raw_df.columns]

sent_features_list = [
    # Non-spectral baselines
    #struct_features,  # already tested in EMIT
    #species,          # already tested in EMIT
    sent_struct_features + species,

    # Spectral alone
    sent_bands,
    sent_indices,
    sent_top_spectral,
    sent_redband,

    # Spectral + height
    sent_bands        + sent_struct_features,
    sent_indices      + sent_struct_features,
    sent_top_spectral + sent_struct_features,
    sent_redband      + sent_struct_features,

    # Spectral + height + species
    sent_bands        + sent_struct_features + species,
    sent_top_spectral + sent_struct_features + species,

    # Spectral + height + interactions (curated where it matters)
    sent_bands        + sent_struct_features + sent_interaction_terms,
    sent_top_spectral + sent_struct_features + sent_top_spectral_interactions,
    sent_redband      + sent_struct_features + sent_redband_interactions,

    # Full specturm of features
    sent_bands        + sent_indices + sent_struct_features + sent_interaction_terms + species
]

## Sentiel. Create plot groups.

In [20]:
# Retain the groups/plot_id for splitting the data based on groups.
plot_groups_sent = None
if 'plot_id' in X_sent_t.columns:
    plot_groups_sent = X_sent_t['plot_id'].copy()
    X_sent_t = X_sent_t.drop(columns=['plot_id'])

if 'plot_id' in X_sent_s.columns:
    X_sent_s = X_sent_s.drop(columns=['plot_id'])

near_zero_sites_sent, high_agb_sites_sent,\
    near_zero_plots_sent, high_agb_plots_sent = \
        get_low_and_high_agb_plots(y_sent,
                                   plot_groups_sent)

grp_sentinel = GROUP_INFO(near_zero_sites_sent,
                          high_agb_sites_sent,
                          near_zero_plots_sent,
                          high_agb_plots_sent,
                          groups=plot_groups_sent,
                          cv=test_cv)

High-AGB threshold  : 1336.17 kg
Near-zero threshold : 0.000052

Near-zero variance plots:
  Big Creek_1               : log var = 0.000036
  Big Creek_4               : log var = 0.000032

High-AGB plots:
  ACA_Acarau Boca           : max AGB = 1468.1 kg
  Arco_del_Espino_18_1      : max AGB = 1564.3 kg
  Arco_del_Espino_18_3      : max AGB = 2403.6 kg
  Arco_del_Espino_18_6      : max AGB = 2403.6 kg
  BAR_Barreto               : max AGB = 2169.6 kg
  BOC_Boca Grande           : max AGB = 5572.3 kg
  Bordo_del_Chile_23_2      : max AGB = 1542.6 kg
  CAE_Caetano               : max AGB = 7364.1 kg
  Desembocadura_Río_Lempa_1_2 : max AGB = 2520.3 kg
  Desembocadura_Río_Lempa_1_3 : max AGB = 1641.7 kg
  Desembocadura_Río_Lempa_1_4 : max AGB = 1594.4 kg
  Desembocadura_Río_Lempa_1_6 : max AGB = 2739.9 kg
  El R’o_5                  : max AGB = 1371.5 kg
  El R’o_6                  : max AGB = 1421.6 kg
  El_Aguacate_24_1          : max AGB = 1627.4 kg
  El_Plan_de_la_Ceiba_8_1   : max AG

In [21]:
assert plot_groups_sent is not None

# TEST COMBINATIONS

In [22]:
test_cv = 5

In [23]:
data_combos= [("EMIT + TANDEMX"    , X_emit_t, y_emit, emit_features_list, grp_emit),
              ("EMIT + SIMARD"     , X_emit_s, y_emit, emit_features_list, grp_emit),
              
              ("SENTINEL + TANDEMX", X_sent_t, y_sent, sent_features_list, grp_sentinel),
              ("SENTINEL + SIMARD" , X_sent_s, y_sent, sent_features_list, grp_sentinel)
             ]
model_list    = ["linear", "rf", "xgboost", "lgbm", "merf"]
is_grids  = [False, True]
is_groups = [True]
#is_groups = [False, True]

In [ ]:
global_experiment_list = {}
experiments_by_category = {}

for (combo_name, X, y, features_list, grp), model, use_grid, use_groups in product(data_combos,
                                                                                   model_list,
                                                                                   is_grids,
                                                                                   is_groups):
    print(f"\n{'='*70}")
    print(f"DATA: {combo_name} | MODEL: {model} | GRID: {use_grid} | GROUPED: {use_groups}")
    print('='*70)
    label = f"{combo_name}, {model}, grid_{"yes" if use_grid else "no"}, GROUPED: {"yes" if use_groups else "no"}"
    exp_result = run_experiment(X, y,
                                is_groups       = use_groups,
                                group_info      = grp,
                                features_list   = features_list,
                                model_type      = model,
                                is_grid         = use_grid,
                                is_stratified   = False,
                                experiment_list = global_experiment_list)
    experiments_by_category[label] = exp_result


DATA: EMIT + TANDEMX | MODEL: linear | GRID: False | GROUPED: False

[1/64]

 EXPERIMENT-1. ,Model: LINEAR REGRESSION, Grouping? No, Hypertuned? No, Features: ['height']
Test R²     : -0.2252
Test RMSE   : 18.66 kg
Train R² (log scale): 0.2534
Train R² (orig scale): -0.0954
Train RMSE  : 17.51 kg
Num rows    : 3104
Num Features: 1

 Cross-validation ---
CV R² mean: 0.2512
CV R² std : 0.0173
CV scores : [0.259 0.25  0.221 0.274 0.252]
 EVALUATION: EXPERIMENT-1. ,Model: LINEAR REGRESSION, Grouping? No, Hypertuned? No, Features: ['height']

Test set:
  R²   : -0.225
  RMSE : 18.66 kg

 ❌ Test R² is negative (-0.225)

Regular CrossValidation:
  Mean   : 0.251
  Std    : 0.017
  Scores : [0.259 0.25  0.221 0.274 0.252]
 ✅ CV mean is positive (0.251)
  ⚠️  Train R² well above test R² (gap: 0.130 > 0.1) — overfitting signal

────────────────────────────────────────────────────────────
VERDICT: ❌ REJECTED

[2/64]

 EXPERIMENT-2. ,Model: RIDGE REGRESSION, Grouping? No, Hypertuned? No, Features

/opt/anaconda3/envs/bkos/lib/python3.14/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/opt/anaconda3/envs/bkos/lib/python3.14/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/opt/anaconda3/envs/bkos/lib/python3.14/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/opt/anaconda3/envs/bkos/lib/python3.14/site-packages/sklearn/utils

Best params: {'n_estimators': 100, 'min_samples_leaf': 4, 'max_samples': 0.6, 'max_features': 0.5, 'max_depth': 5}

 EXPERIMENT-4. ,Model: RANDOM FOREST, Grouping? No, Hypertuned? Yes, Features: ['EMIT bands (280)']
Test R²     : 0.1242
Test RMSE   : 15.78 kg
Train R² (log scale): 0.3620
Train R² (orig scale): 0.1621
Train RMSE  : 15.32 kg
Num rows    : 3104
Num Features: 280

 Cross-validation ---
CV R² mean: 0.3429
CV R² std : 0.0186
CV scores : [0.36  0.314 0.328 0.35  0.362]
 EVALUATION: EXPERIMENT-4. ,Model: RANDOM FOREST, Grouping? No, Hypertuned? Yes, Features: ['EMIT bands (280)']

Test set:
  R²   : 0.124
  RMSE : 15.78 kg

 ✅ Test R² is positive (0.124)

Regular CrossValidation:
  Mean   : 0.343
  Std    : 0.019
  Scores : [0.36  0.314 0.328 0.35  0.362]
 ✅ CV mean is positive (0.343)
  ✅ Train/test R² gap acceptable (gap: 0.038)

────────────────────────────────────────────────────────────
VERDICT: ✅ ACCEPTABLE

Top 10 feature importances:
  EMIT_R596                        

/opt/anaconda3/envs/bkos/lib/python3.14/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/opt/anaconda3/envs/bkos/lib/python3.14/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/opt/anaconda3/envs/bkos/lib/python3.14/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/opt/anaconda3/envs/bkos/lib/python3.14/site-packages/sklearn/utils

Best params: {'n_estimators': 100, 'min_samples_leaf': 4, 'max_samples': 0.6, 'max_features': 'sqrt', 'max_depth': 10}

 EXPERIMENT-7. ,Model: RANDOM FOREST, Grouping? No, Hypertuned? Yes, Features: ['EMIT bands (280)', 'height']
Test R²     : 0.1368
Test RMSE   : 15.67 kg
Train R² (log scale): 0.4030
Train R² (orig scale): 0.1827
Train RMSE  : 15.13 kg
Num rows    : 3104
Num Features: 281

 Cross-validation ---
CV R² mean: 0.3742
CV R² std : 0.0299
CV scores : [0.426 0.345 0.344 0.376 0.38 ]
 EVALUATION: EXPERIMENT-7. ,Model: RANDOM FOREST, Grouping? No, Hypertuned? Yes, Features: ['EMIT bands (280)', 'height']

Test set:
  R²   : 0.137
  RMSE : 15.67 kg

 ✅ Test R² is positive (0.137)

Regular CrossValidation:
  Mean   : 0.374
  Std    : 0.030
  Scores : [0.426 0.345 0.344 0.376 0.38 ]
 ✅ CV mean is positive (0.374)
  ✅ Train/test R² gap acceptable (gap: 0.046)

────────────────────────────────────────────────────────────
VERDICT: ✅ ACCEPTABLE

Top 10 feature importances:
  height   

/opt/anaconda3/envs/bkos/lib/python3.14/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/opt/anaconda3/envs/bkos/lib/python3.14/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/opt/anaconda3/envs/bkos/lib/python3.14/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/opt/anaconda3/envs/bkos/lib/python3.14/site-packages/sklearn/utils

Best params: {'n_estimators': 100, 'min_samples_leaf': 2, 'max_samples': 0.6, 'max_features': 0.3, 'max_depth': 10}

 EXPERIMENT-10. ,Model: RANDOM FOREST, Grouping? No, Hypertuned? Yes, Features: ['EMIT bands (280)', 'height', 'species']
Test R²     : 0.1431
Test RMSE   : 15.61 kg
Train R² (log scale): 0.4569
Train R² (orig scale): 0.2005
Train RMSE  : 14.96 kg
Num rows    : 3104
Num Features: 284

 Cross-validation ---
CV R² mean: 0.4106
CV R² std : 0.0363
CV scores : [0.455 0.361 0.374 0.429 0.433]
 EVALUATION: EXPERIMENT-10. ,Model: RANDOM FOREST, Grouping? No, Hypertuned? Yes, Features: ['EMIT bands (280)', 'height', 'species']

Test set:
  R²   : 0.143
  RMSE : 15.61 kg

 ✅ Test R² is positive (0.143)

Regular CrossValidation:
  Mean   : 0.411
  Std    : 0.036
  Scores : [0.455 0.361 0.374 0.429 0.433]
 ✅ CV mean is positive (0.411)
  ✅ Train/test R² gap acceptable (gap: 0.057)

────────────────────────────────────────────────────────────
VERDICT: ✅ ACCEPTABLE

Top 10 feature imp

/opt/anaconda3/envs/bkos/lib/python3.14/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/opt/anaconda3/envs/bkos/lib/python3.14/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/opt/anaconda3/envs/bkos/lib/python3.14/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/opt/anaconda3/envs/bkos/lib/python3.14/site-packages/sklearn/utils

# FIND THE BEST EXPERIMENT SO FAR.

## SUMMARY OF EXPERIMENTS

In [ ]:
%run Model_functions.ipynb
best_results = filter_best_experiments(global_experiment_list)
tab_df = tabulate_results(best_results)

## BEST ONE

In [ ]:
%run Model_functions.ipynb
best_result = best_results[0][1]
print_experiment(best_result)